# Indian Pines: agregación espectral y características espaciales
**Autor del notebook original: Camilo Acevedo-Correa.**

Esta versión organiza el procesamiento y corrige errores de implementación. Conserva los nueve intervalos espectrales del original. Con el mapeo estándar de Indian Pines corregido (200 canales), B9 no tiene muestras: se generan **8 bandas y 24 características**. Con el cubo completo de 220 canales se pueden generar **9 bandas y 27 características**, aunque la región de absorción requiere cautela científica.

La media por intervalo aproxima una MSI; no reproduce la respuesta radiométrica ni las resoluciones de Landsat 8. Aquí se extraen características; no se entrena ni evalúa un clasificador.


## 1. Preparar el entorno
En Colab, ejecutar la siguiente celda descarga el código de este repositorio e instala las dependencias. Localmente, instalar `requirements.txt` y abrir este notebook desde la raíz del repositorio.

In [ ]:
import sys
from pathlib import Path
import subprocess
import importlib.util

if 'google.colab' in sys.modules:
    project = Path('/content/indian-pines-hsi-to-msi')
    if not project.exists():
        subprocess.run(['git', 'clone', 'https://github.com/camiloace/indian-pines-hsi-to-msi.git', str(project)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(project/'requirements.txt')], check=True)
else:
    project = Path.cwd()
if not (project/'hsi_processing.py').exists():
    raise FileNotFoundError('Abra el notebook desde la raíz del repositorio')
sys.path.insert(0, str(project))


## 2. Proporcionar los datos
Descarga [Indian_pines_corrected.mat desde la copia pública de Hugging Face](https://huggingface.co/datasets/danaroth/indian_pines/resolve/main/Indian_pines_corrected.mat?download=true) y colócalo en `data/`. Esta copia es mantenida por terceros; la descarga verificada coincide byte por byte con el MAT usado para validar el notebook (145×145×200).

SHA-256: `ec2f8808710919d566f70f0d4aa885aae1ddfd42b734aba71c5e12ca65450939`.

La página de EHU citada al final se conserva como referencia de la fuente original, pero devuelve HTTP 403 en las comprobaciones realizadas. El archivo MAT no se redistribuye en este repositorio.

En Colab, ejecutar la celda siguiente permite cargar un `.mat` si no existe aún. Para 220 canales, ajustar `MAT_PATH` y `MAT_KEY` al archivo utilizado.


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import loadmat, savemat
from hsi_processing import wavelengths, standard_channels, aggregate_bands, spatial_features, normalize_image

MAT_PATH = project/'data/Indian_pines_corrected.mat'
MAT_KEY = 'indian_pines_corrected'
if not MAT_PATH.exists() and 'google.colab' in sys.modules:
    from google.colab import files
    uploaded = files.upload()
    if MAT_PATH.name in uploaded:
        MAT_PATH.write_bytes(uploaded[MAT_PATH.name])
if not MAT_PATH.exists():
    raise FileNotFoundError(f'Coloque el dataset en {MAT_PATH}')
cube = loadmat(MAT_PATH)[MAT_KEY]
print('HSI:', cube.shape, cube.dtype)


## 3. Alinear los canales
**Supuesto explícito:** para 200 bandas se emplea la convención que elimina los canales originales 104–108, 150–163 y 220 (numeración desde 1). El nombre y la forma del archivo no prueban este mapeo. Si su cubo fue modificado, reemplace `channel_ids` por los identificadores originales, en el orden exacto del cubo.

El archivo `wavelenght.csv` contiene las longitudes de onda AVIRIS. Verifica sus valores frente a la tabla de calibración incluida en la [publicación de Purdue PURR](https://purr.purdue.edu/publications/1947/1). El enlace alternativo para descargar el MAT corregido de 200 bandas está en la sección 2.


In [ ]:
channel_ids = standard_channels(cube.shape[2])
wave = wavelengths(project/'data/wavelenght.csv', channel_ids)
assert len(wave) == cube.shape[2]
print('Canales alineados:', len(wave))


## 4. Promediar las bandas de cada intervalo
`missing='skip'` omite explícitamente los intervalos sin muestras. Use `missing='error'` para exigir cobertura de los nueve intervalos. No se interpola ni sustituye B9 por un canal fuera de su intervalo.


In [ ]:
msi, selection = aggregate_bands(cube, wave, missing='skip')
labels = [r['band'] for r in selection if r['included']]
display(pd.DataFrame(selection)[['band','lower_nm','upper_nm','count','included']])
print('MSI:', msi.shape)


## 5. Entropía local y morfología
La entropía usa normalización min–max global, cuantización explícita a 8 bits y un disco de radio 4 píxeles. Cada mapa de entropía se divide por su máximo; un mapa constante conserva ceros.

La morfología usa, por banda, dos erosiones, apertura, dos dilataciones y cierre/apertura por área con umbral de 1000 píxeles. El elemento estructurante es de 5×5. Estos parámetros siguen el notebook original y pueden eliminar regiones pequeñas; deben evaluarse según la tarea.


In [ ]:
entropic, morph = spatial_features(msi)
features = np.concatenate([msi, entropic, morph], axis=2)
assert np.isfinite(features).all()
print('MSI:', msi.shape, 'Entropía:', entropic.shape, 'Morfología:', morph.shape)
print('Características:', features.shape)


## 6. Visualización
El compuesto RGB usa B4, B3 y B2. Las escalas de visualización se ajustan para mostrar contraste; no representan reflectancia calibrada.


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16,4))
rgb = msi[:, :, [labels.index('B4'),labels.index('B3'),labels.index('B2')]]
rgb = np.stack([normalize_image(rgb[:,:,i]) for i in range(3)], axis=2)
axes[0].imshow(rgb); axes[0].set_title('Compuesto RGB')
for ax, image, title in zip(axes[1:], [msi[:,:,0],entropic[:,:,0],morph[:,:,0]], ['MSI B1','Entropía B1','Morfología B1']):
    ax.imshow(image, cmap='viridis'); ax.set_title(title)
for ax in axes: ax.axis('off')
fig.tight_layout()
plt.show()


## 7. Guardar los resultados
Se escriben archivos separados para evitar sobrescrituras entre MSI y características. El JSON registra la selección, los canales originales y los parámetros. Los 24/27 canales finales mezclan intensidades y entropía normalizada: antes de entrenar un modelo, ajustar el escalado con los datos de entrenamiento solamente.


In [ ]:
output = project/'results'
output.mkdir(exist_ok=True)
count = msi.shape[2]
savemat(output/f'IP_MSI_{count}.mat', {'msi':msi})
savemat(output/f'IP_FEATURES_{features.shape[2]}.mat', {'features':features, 'entropy':entropic, 'morphology':morph})
metadata = dict(input_shape=list(cube.shape), input_key=MAT_KEY, channel_ids=channel_ids.tolist(),
                selection=selection, channel_order=['MSI','entropy','morphology'],
                entropy_radius=4, footprint_size=5, iterations=2, area_threshold=1000,
                mapping_assumption='standard_channels; verify against dataset provenance')
(output/'processing_metadata.json').write_text(json.dumps(metadata,ensure_ascii=False,indent=2), encoding='utf-8')
fig.savefig(output/'preview.png', dpi=150, bbox_inches='tight')
print('Resultados guardados en', output)


## Referencias y atribución
Autor original: **Camilo Acevedo-Correa**. Ver `notebook-original-sin-salidas.ipynb` para el código original, sin sus salidas guardadas.

1. [Landsat 8 — NASA](https://science.nasa.gov/mission/landsat-8/).
2. [Indian Pines — EHU](https://www.ehu.eus/ccwintco/index.php/Hyperspectral_Remote_Sensing_Scenes#Indian_Pines).
3. [Image Processing with Python: Working with Entropy](https://towardsdatascience.com/image-processing-with-python-working-with-entropy-b05e9c84fc36).
4. [Morphological Operations — j-manansala](https://github.com/j-manansala/morphological/blob/main/Morphological%20Operations.ipynb).
5. [Baumgardner, Biehl y Landgrebe (2015), datos y calibración](https://doi.org/10.4231/R7RX991C).
